# NB_08 — GDT × SMuRF Channel Assignment v1.0

This notebook combines the published SMuRF / µMUX architecture in `SOURCE_05`
with the implementation-facing PySMuRF assignment rules in `SOURCE_06`.

The question is deliberately narrower than “can GDT apply to SMuRF?”:

> Which admissible readout states are actually specified by the evidence, and
> which—if any—satisfy the hypotheses needed for a direct General Divisor
> Theorem application?

A direct GDT claim requires physically specified meanings for

\[
n,\quad m,\quad a,\quad N
\]

with

\[
n \equiv a \pmod m
\]

and

\[
\gcd(n,N)=1.
\]

No arbitrary integerization, unit conversion, or factor rule may be introduced
solely to make the theorem fit.


## 1. Locate repository and inputs


In [ ]:
from pathlib import Path
import json
import math
import subprocess
import sys

import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"

def find_repo_root():
    start = Path.cwd().resolve()
    candidates = [
        start, *start.parents,
        Path("/content/sensors-becker"),
        Path("/home/dan/sensors-becker"),
        Path.home() / "sensors-becker",
    ]
    for c in candidates:
        if c.is_dir() and (c / "engineering_navigator").is_dir():
            return c

    if Path("/content").exists():
        target = Path("/content/sensors-becker")
        if not target.exists():
            subprocess.run(["git", "clone", REPOSITORY_URL, str(target)], check=True)
        if (target / "engineering_navigator").is_dir():
            return target

    raise FileNotFoundError("Could not locate sensors-becker")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SOURCE_DIR = ROOT / "engineering_navigator" / "multiplexed_readout" / "source_records"
S05 = SOURCE_DIR / "SOURCE_05_smurf_multiplexed_readout.yaml"
S06 = SOURCE_DIR / "SOURCE_06_pysmurf_channel_assignment.yaml"
OBJECT_PATH = ROOT / "engineering_navigator" / "engineering_objects" / "multiplexed_readout.yaml"

OUTPUT_DIR = ROOT / "engineering_navigator" / "multiplexed_readout" / "gdt_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [S05, S06]:
    if not p.exists():
        raise FileNotFoundError(p)

source_05 = yaml.safe_load(S05.read_text(encoding="utf-8"))
source_06 = yaml.safe_load(S06.read_text(encoding="utf-8"))

print("Repository:", ROOT)
print("SOURCE_05:", source_05.get("title"))
print("SOURCE_06:", source_06.get("title"))
print("Engineering Object exists:", OBJECT_PATH.exists())


## 2. Evidence checkpoint


In [ ]:
checkpoint = {
    "SOURCE_05": {
        "source_id": source_05.get("source_id"),
        "gdt_status": source_05.get("gdt_applicability", {}).get("status"),
    },
    "SOURCE_06": {
        "source_id": source_06.get("source_id"),
        "gdt_status": source_06.get("gdt_applicability", {}).get("status"),
    },
}

print(json.dumps(checkpoint, indent=2))


## 3. Candidate engineering states

The combined evidence gives genuine discrete structure, but discrete structure
alone is not enough for GDT.

The audit distinguishes:

- **integer state present**
- **natural modulus present**
- **physical residue-class rule present**
- **physical coprimality/factor-exclusion rule present**
- **direct GDT ready**


In [ ]:
candidates = [
    {
        "candidate_id": "SMURF_GDT_001",
        "name": "analysis_subband_index",
        "evidence_sources": ["SOURCE_05", "SOURCE_06"],
        "integer_state_n": True,
        "natural_modulus_m": True,
        "candidate_m": 128,
        "physical_residue_rule": False,
        "physical_coprimality_rule": False,
        "direct_gdt_ready": False,
        "engineering_basis": (
            "SOURCE_05 specifies 128 overlapping digital analysis sub-bands; "
            "SOURCE_06 confirms assignment to integer subband indices."
        ),
        "missing": (
            "No documented repeating allowed-residue rule and no factor-based "
            "index exclusion."
        ),
    },
    {
        "candidate_id": "SMURF_GDT_002",
        "name": "channel_number_within_subband",
        "evidence_sources": ["SOURCE_06"],
        "integer_state_n": True,
        "natural_modulus_m": True,
        "candidate_m": 4,
        "physical_residue_rule": False,
        "physical_coprimality_rule": False,
        "direct_gdt_ready": False,
        "engineering_basis": (
            "PySMuRF assign_channels returns integer channel numbers and defaults "
            "to four channels per subband."
        ),
        "missing": (
            "Bounded capacity of four channels per subband is not itself a residue "
            "condition, and no gcd exclusion is documented."
        ),
    },
    {
        "candidate_id": "SMURF_GDT_003",
        "name": "resonator_grouping",
        "evidence_sources": ["SOURCE_05"],
        "integer_state_n": True,
        "natural_modulus_m": True,
        "candidate_m": 4,
        "physical_residue_rule": False,
        "physical_coprimality_rule": False,
        "direct_gdt_ready": False,
        "engineering_basis": (
            "SOURCE_05 reports four resonator groups/sub-bands per 66-channel chip."
        ),
        "missing": (
            "No explicit modular placement formula or factor-exclusion law."
        ),
    },
    {
        "candidate_id": "SMURF_GDT_004",
        "name": "tracked_harmonic_order",
        "evidence_sources": ["SOURCE_05"],
        "integer_state_n": True,
        "natural_modulus_m": True,
        "candidate_m": 3,
        "physical_residue_rule": False,
        "physical_coprimality_rule": False,
        "direct_gdt_ready": False,
        "engineering_basis": (
            "Tracked Fourier harmonic indices n = 1, 2, 3 are explicit discrete states."
        ),
        "missing": (
            "The first-three-harmonics tracking approximation does not specify a "
            "residue class or coprimality condition."
        ),
    },
]

candidate_df = pd.DataFrame(candidates)
candidate_df


## 4. Physical exclusion rules


In [ ]:
exclusions = [
    {
        "exclusion_id": "EXCL_001",
        "name": "close_resonator_collision",
        "source": "SOURCE_05",
        "rule_type": "frequency_distance_threshold",
        "rule": "adjacent resonators near or closer than roughly one linewidth (~100 kHz) are commonly untrackable",
        "factor_based": False,
        "gdt_role": "physical exclusion, but not gcd(n,N)=1",
    },
    {
        "exclusion_id": "EXCL_002",
        "name": "pysmurf_minimum_offset",
        "source": "SOURCE_06",
        "rule_type": "frequency_distance_threshold",
        "rule": "ignore both resonators if separation < 0.1 MHz",
        "factor_based": False,
        "gdt_role": "implementation exclusion, but not gcd(n,N)=1",
    },
    {
        "exclusion_id": "EXCL_003",
        "name": "band_edge_exclusion",
        "source": "SOURCE_05",
        "rule_type": "frequency_window",
        "rule": "resonators too near 500 MHz band edges cannot be tracked",
        "factor_based": False,
        "gdt_role": "physical window constraint, not a factor exclusion",
    },
    {
        "exclusion_id": "EXCL_004",
        "name": "tracking_quality_cut",
        "source": "SOURCE_05 + SOURCE_06",
        "rule_type": "quality_state",
        "rule": "disable channels that fail tracking or quality limits",
        "factor_based": False,
        "gdt_role": "admissibility criterion, but not a documented arithmetic divisor rule",
    },
]

exclusion_df = pd.DataFrame(exclusions)
exclusion_df


## 5. GDT hypothesis audit

A candidate is marked direct-GDT-ready only where all four conditions are true:

1. a physically meaningful integer state \(n\),
2. a natural modulus \(m\),
3. a documented physical residue condition \(n \equiv a \pmod m\),
4. a documented physical factor exclusion \(\gcd(n,N)=1\).


In [ ]:
audit_df = candidate_df[
    [
        "candidate_id",
        "name",
        "integer_state_n",
        "natural_modulus_m",
        "physical_residue_rule",
        "physical_coprimality_rule",
        "direct_gdt_ready",
    ]
].copy()

audit_df


## 6. Direct GDT evaluation gate


In [ ]:
ready_df = candidate_df[candidate_df["direct_gdt_ready"] == True].copy()

GDT_RESULT_COLUMNS = [
    "candidate_id", "name", "N", "m", "a",
    "d", "R", "minimal_period",
    "accepted_per_period", "correction_factor"
]

gdt_results_df = pd.DataFrame(columns=GDT_RESULT_COLUMNS)

if ready_df.empty:
    print("No physically justified direct GDT mapping is available.")
    print("No theorem calculation will be manufactured from incomplete hypotheses.")
else:
    print("Direct-GDT-ready candidates:", len(ready_df))

gdt_results_df


## 7. Admissible-state specification available without GDT


In [ ]:
specified_states = [
    {
        "state_class": "assigned_readout_channel",
        "specified_by": "SOURCE_06",
        "admissibility_basis": "resonator frequency mapped by assign_channels",
        "gdt_required": False,
    },
    {
        "state_class": "frequency-separated_resonator_pair",
        "specified_by": "SOURCE_05 + SOURCE_06",
        "admissibility_basis": "pairwise frequency-spacing threshold",
        "gdt_required": False,
    },
    {
        "state_class": "in-band_resonator",
        "specified_by": "SOURCE_05",
        "admissibility_basis": "usable readout-frequency windows / band-edge exclusion",
        "gdt_required": False,
    },
    {
        "state_class": "tracking-qualified_channel",
        "specified_by": "SOURCE_05 + SOURCE_06",
        "admissibility_basis": "tracking and quality limits",
        "gdt_required": False,
    },
]

specified_states_df = pd.DataFrame(specified_states)
specified_states_df


## 8. Combined conclusion


In [ ]:
status = {
    "notebook_id": "NB_08_GDT_SMURF_CHANNEL_ASSIGNMENT",
    "version": "1.0.0",
    "engineering_domain": "TES microwave SQUID multiplexed readout",
    "evidence_sources": ["SOURCE_05", "SOURCE_06"],
    "candidate_mapping_count": int(len(candidate_df)),
    "physically_justified_direct_gdt_mapping_count": int(len(ready_df)),
    "direct_gdt_claim_allowed": bool(len(ready_df) > 0),
    "specified_non_gdt_admissible_state_classes": specified_states_df["state_class"].tolist(),
    "current_conclusion": (
        "SOURCE_05 and SOURCE_06 specify genuine discrete readout states and "
        "multiple physical admissibility/exclusion rules. The implementation-facing "
        "evidence strengthens the engineering specification, but the documented "
        "assignment and exclusion rules still do not supply both a physical residue-class "
        "condition and a coprimality/factor-exclusion condition. Direct application of "
        "the General Divisor Theorem therefore remains unestablished."
    ),
    "engineering_value": (
        "The navigator can still specify admissible readout states from documented "
        "frequency, assignment, band-edge, and tracking constraints without forcing "
        "those constraints into GDT."
    ),
    "next_evidence_target": (
        "Inspect lower-level firmware, frequency-plan, mask-layout, clocking, aliasing, "
        "or synchronization rules only where they expose an exact repeating integer "
        "allocation or factor-based exclusion."
    ),
}

print(json.dumps(status, indent=2))


## 9. Write reproducible audit artifacts


In [ ]:
candidate_df.to_csv(OUTPUT_DIR / "candidate_gdt_mappings.csv", index=False)
audit_df.to_csv(OUTPUT_DIR / "gdt_hypothesis_audit.csv", index=False)
gdt_results_df.to_csv(OUTPUT_DIR / "gdt_readout_results.csv", index=False)
exclusion_df.to_csv(OUTPUT_DIR / "physical_exclusion_audit.csv", index=False)
specified_states_df.to_csv(OUTPUT_DIR / "specified_admissible_states.csv", index=False)

(OUTPUT_DIR / "gdt_smurf_status.json").write_text(
    json.dumps(status, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Wrote:")
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print("-", p.relative_to(ROOT))


## Result

The useful engineering statement at this reading point is not “GDT applies.”

It is:

> SOURCE_05 + SOURCE_06 specify real admissible readout states.  
> Where those states are governed by frequency-distance, band-window, assignment,
> and tracking constraints, the navigator preserves those constraints directly.  
> A GDT specialization remains gated on evidence for an actual residue-class rule
> together with an actual factor-exclusion rule.

This is a successful applicability audit because the theorem boundary is itself
machine-readable and reproducible.
